# Module 5 Project Solution: Explainability, Confidence, OOD, and Robustness


> **Colab setup:** This solution notebook uses only the default open-source Python stack available in Google Colab: NumPy, Matplotlib, and scikit-learn. No `pip install`, no API keys, and no external authorization are required.
>
> Running all cells produces the complete reference output, including plots and metrics.


## Project map
This project turns XAI and robustness into measurable signals: confidence, entropy, occlusion sensitivity, shift behavior, and slice-level performance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(7)
digits = load_digits()
X_img = digits.images / 16.0
X = X_img.reshape(len(X_img), -1)
y = digits.target

X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X, y, X_img, test_size=0.25, random_state=7, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
clf = LogisticRegression(max_iter=800)
clf.fit(X_train_s, y_train)
proba = clf.predict_proba(X_test_s)
pred = proba.argmax(axis=1)
print("accuracy:", accuracy_score(y_test, pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred)).plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix")
plt.show()


In [ ]:
confidence = proba.max(axis=1)
entropy = -np.sum(proba * np.log(proba + 1e-12), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(confidence, bins=20); axes[0].set_title("confidence")
axes[1].hist(entropy, bins=20); axes[1].set_title("entropy")
plt.show()


In [ ]:
noise = np.random.normal(0, 0.45, size=img_test.shape)
ood_img = np.clip(img_test + noise, 0, 1)
ood_X = ood_img.reshape(len(ood_img), -1)
ood_proba = clf.predict_proba(scaler.transform(ood_X))
ood_conf = ood_proba.max(axis=1)

plt.hist(confidence, bins=20, alpha=0.6, label="in distribution")
plt.hist(ood_conf, bins=20, alpha=0.6, label="shifted/OOD-like")
plt.legend()
plt.title("Confidence under distribution shift")
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(img_test[i], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(ood_img[i], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("ID")
axes[1, 0].set_ylabel("shift")
plt.show()


In [ ]:
idx = 3
image = img_test[idx].copy()
target = clf.predict(scaler.transform(image.reshape(1, -1)))[0]

def score_image(image, target):
    p = clf.predict_proba(scaler.transform(image.reshape(1, -1)))[0]
    return p[target]

base_score = score_image(image, target)
heat = np.zeros_like(image)
for r in range(0, 8, 2):
    for c in range(0, 8, 2):
        occluded = image.copy()
        occluded[r:r+2, c:c+2] = 0.0
        heat[r:r+2, c:c+2] = base_score - score_image(occluded, target)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image, cmap="gray"); axes[0].set_title(f"digit {target}")
axes[1].imshow(heat, cmap="inferno"); axes[1].set_title("occlusion importance")
for ax in axes: ax.axis("off")
plt.show()


In [ ]:
brightness = X_test.mean(axis=1)
threshold = np.median(brightness)
low = brightness <= threshold
high = brightness > threshold
acc_low = accuracy_score(y_test[low], pred[low])
acc_high = accuracy_score(y_test[high], pred[high])

plt.bar(["darker half", "brighter half"], [acc_low, acc_high])
plt.ylim(0, 1)
plt.title("Slice accuracy by brightness")
plt.show()
print("darker accuracy:", acc_low)
print("brighter accuracy:", acc_high)
